# Qwen2.5-VL EGTEA Frame-by-Frame Analysis Demo

Minimal notebook to test Qwen2.5-VL on one randomly selected EGTEA video. It samples frames from the video, runs a visual-language prompt on each frame, and saves per-frame analysis to JSONL/CSV.

## 1. Imports and configuration

In [ ]:
from __future__ import annotations

import json
import random
import sys
import time
import traceback
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

try:
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except Exception:
    torch = None
    DEVICE = "cpu"


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for path in [start, *start.parents]:
        if (path / "notebooks").exists() and (path / "scripts").exists():
            return path
    return Path.cwd().resolve()


REPO_ROOT = find_repo_root()
VIDEO_ROOT = Path(r"C:\Users\18447\Detector\data\egtea_gaze_plus\video_clips\cropped_clips")
OUTPUT_ROOT = REPO_ROOT / "outputs" / "qwen25vl_egtea_frame_analysis"
FRAME_OUTPUT_DIR = OUTPUT_ROOT / "sampled_frames"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
FRAME_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"
LOCAL_FILES_ONLY = True  # use cached local weights; set False if you want HF download fallback
RANDOM_SEED = 42
N_FRAMES = 12
MIN_SECONDS_BETWEEN_FRAMES = 0.25
MAX_NEW_TOKENS = 180

FRAME_PROMPT = """
Analyze this egocentric kitchen video frame. Be concise and structured.
Return:
1. visible hands and hand-object interactions
2. key objects and containers
3. current action or step if inferable
4. visual quality issues: blur, occlusion, motion, lighting, clutter
5. whether this frame looks useful for auto-labeling object masks: yes/no/uncertain and why
""".strip()

print(f"Repo root : {REPO_ROOT}")
print(f"Video root: {VIDEO_ROOT}")
print(f"Output   : {OUTPUT_ROOT}")
print(f"Device   : {DEVICE}")
print(f"Model    : {MODEL_ID}")

## 2. Select one random EGTEA video

In [ ]:
VIDEO_EXTENSIONS = {".mp4", ".avi", ".mov", ".mkv"}


def list_videos(root: Path) -> list[Path]:
    if not root.exists():
        return []
    return sorted(p for p in root.rglob("*") if p.is_file() and p.suffix.lower() in VIDEO_EXTENSIONS)


videos = list_videos(VIDEO_ROOT)
if not videos:
    raise FileNotFoundError(f"No videos found under {VIDEO_ROOT}")

rng = random.Random(RANDOM_SEED)
video_path = rng.choice(videos)
print(f"Found {len(videos)} videos")
print(f"Selected: {video_path}")

## 3. Sample frames from the video

In [ ]:
def sample_video_frames(video_path: Path, n_frames: int, seed: int) -> list[dict]:
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"Could not open video: {video_path}")
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    fps = float(cap.get(cv2.CAP_PROP_FPS) or 30.0)
    duration = frame_count / fps if fps > 0 else 0.0
    if frame_count <= 0:
        cap.release()
        raise RuntimeError(f"Video has no readable frames: {video_path}")

    rng = random.Random(seed)
    min_gap = max(1, int(MIN_SECONDS_BETWEEN_FRAMES * fps))
    chosen = []
    attempts = 0
    while len(chosen) < min(n_frames, frame_count) and attempts < n_frames * 50:
        attempts += 1
        idx = rng.randrange(frame_count)
        if all(abs(idx - old) >= min_gap for old in chosen):
            chosen.append(idx)
    if len(chosen) < n_frames:
        chosen = sorted(set(chosen + rng.sample(range(frame_count), min(n_frames - len(chosen), frame_count))))[:n_frames]
    chosen = sorted(chosen)

    rows = []
    for out_i, frame_idx in enumerate(chosen):
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        ok, frame_bgr = cap.read()
        if not ok or frame_bgr is None:
            continue
        out_path = FRAME_OUTPUT_DIR / f"frame_{out_i:03d}_f{frame_idx}.jpg"
        cv2.imwrite(str(out_path), frame_bgr)
        rows.append({
            "frame_number": out_i,
            "frame_index": frame_idx,
            "timestamp_sec": frame_idx / fps if fps > 0 else None,
            "frame_path": out_path,
        })
    cap.release()
    print(f"Video fps={fps:.2f}, frames={frame_count}, duration={duration:.2f}s")
    print(f"Sampled {len(rows)} frames to {FRAME_OUTPUT_DIR}")
    return rows


sampled = sample_video_frames(video_path, N_FRAMES, RANDOM_SEED)
pd.DataFrame([{**r, "frame_path": str(r["frame_path"])} for r in sampled])

## 4. Preview sampled frames

In [ ]:
cols = 4
rows = (len(sampled) + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(16, max(4, rows * 3)))
axes = axes.flatten() if hasattr(axes, "flatten") else [axes]
for ax, item in zip(axes, sampled):
    img = Image.open(item["frame_path"]).convert("RGB")
    ax.imshow(img)
    ax.set_title(f"#{item['frame_number']} t={item['timestamp_sec']:.2f}s")
    ax.axis("off")
for ax in axes[len(sampled):]:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 5. Load Qwen2.5-VL

In [ ]:
qwen_model = None
qwen_processor = None
qwen_load_error = ""

try:
    from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

    dtype = torch.float16 if DEVICE == "cuda" else torch.float32
    qwen_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        MODEL_ID,
        torch_dtype=dtype,
        device_map="auto" if DEVICE == "cuda" else None,
        local_files_only=LOCAL_FILES_ONLY,
    ).eval()
    if DEVICE == "cpu":
        qwen_model = qwen_model.to(DEVICE)
    qwen_processor = AutoProcessor.from_pretrained(MODEL_ID, local_files_only=LOCAL_FILES_ONLY)
    print("Qwen2.5-VL loaded")
except Exception as exc:
    qwen_load_error = f"{type(exc).__name__}: {exc}"
    print("Failed to load Qwen2.5-VL")
    print(qwen_load_error)
    traceback.print_exc(limit=2)
    print("If needed: pip install -U transformers accelerate qwen-vl-utils")

## 6. Frame analysis function

In [ ]:
def analyze_frame_with_qwen(image_path: Path, prompt: str) -> dict:
    if qwen_model is None or qwen_processor is None:
        return {"analysis": "", "error": qwen_load_error or "model not loaded", "seconds": 0.0}
    image = Image.open(image_path).convert("RGB")
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": prompt},
            ],
        }
    ]
    started = time.perf_counter()
    try:
        text = qwen_processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = qwen_processor(text=[text], images=[image], return_tensors="pt")
        inputs = inputs.to(DEVICE)
        with torch.inference_mode():
            generated_ids = qwen_model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
        generated_trimmed = [out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)]
        output_text = qwen_processor.batch_decode(generated_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]
        return {"analysis": output_text.strip(), "error": "", "seconds": time.perf_counter() - started}
    except Exception as exc:
        return {"analysis": "", "error": f"{type(exc).__name__}: {exc}", "seconds": time.perf_counter() - started}


print(FRAME_PROMPT)

## 7. Run frame-by-frame analysis

In [ ]:
records = []
jsonl_path = OUTPUT_ROOT / "qwen25vl_frame_analysis.jsonl"

with jsonl_path.open("w", encoding="utf-8") as fh:
    for item in sampled:
        print(f"Analyzing frame {item['frame_number']} at {item['timestamp_sec']:.2f}s")
        result = analyze_frame_with_qwen(item["frame_path"], FRAME_PROMPT)
        record = {
            "video_path": str(video_path),
            "frame_number": item["frame_number"],
            "frame_index": item["frame_index"],
            "timestamp_sec": item["timestamp_sec"],
            "frame_path": str(item["frame_path"]),
            "prompt": FRAME_PROMPT,
            **result,
        }
        records.append(record)
        fh.write(json.dumps(record, ensure_ascii=False) + "\n")
        print(result["analysis"][:500] if result["analysis"] else result["error"])
        print("-" * 80)

df = pd.DataFrame(records)
csv_path = OUTPUT_ROOT / "qwen25vl_frame_analysis.csv"
df.to_csv(csv_path, index=False)
print(f"Saved JSONL: {jsonl_path}")
print(f"Saved CSV  : {csv_path}")
display(df[["frame_number", "timestamp_sec", "seconds", "error", "analysis"]])

## 8. Create visual report

In [ ]:
report_md = OUTPUT_ROOT / "qwen25vl_frame_analysis_report.md"
with report_md.open("w", encoding="utf-8") as fh:
    fh.write(f"# Qwen2.5-VL EGTEA Frame Analysis\n\n")
    fh.write(f"Video: `{video_path}`\n\n")
    for record in records:
        fh.write(f"## Frame {record['frame_number']} - {record['timestamp_sec']:.2f}s\n\n")
        fh.write(f"Image: `{record['frame_path']}`\n\n")
        if record["error"]:
            fh.write(f"Error: `{record['error']}`\n\n")
        else:
            fh.write(record["analysis"] + "\n\n")
print(f"Saved report: {report_md}")